# [Ablation arm B] 2-Tower (텍스트 + 과거주가) — Colab · A100

`lib/train_colab.py`(**arm A, 텍스트만**)와 **동일 fold·class-weighted focal loss·macro-F1**으로 비교하는 두 번째 팔.

**핵심(누수 방어):** 과거 주가 ≠ 미래 주가. 정답(Class 2·3)을 만든 *미래* 주가는 입력 금지지만,
**댓글 작성 시각 *직전*까지의 시간봉**은 작성자·인간 평가자 모두가 보는 정보라 입력에 넣어도 누수가 아니다
(strict 미래 컷오프 `assert`로 보장).

모델: KcELECTRA `[CLS]` ⊕ 가격 bi-GRU(masked-mean) → concat → MLP head.

## 0. 환경 준비 & 데이터 업로드

In [ ]:
!pip install -q -U transformers accelerate scikit-learn
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

업로드 파일 **4개**: 학습/평가 CSV 2개 + 시간별 가격 CSV 2개
- `tsla_train_enriched_final.csv`, `nvda_eval_final.csv`
- `TSLA_prices_1h.csv`, `NVDA_prices_1h.csv`

In [ ]:
from google.colab import files
up = files.upload()
print("업로드됨:", list(up))
# 업로드는 현재 폴더(.)에 저장됨 → 아래 CONFIG의 DATA_DIR/PRICE_DIR='.'

## 1. 설정 (CONFIG)

In [ ]:
import os, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModel, Trainer, TrainingArguments,
                          EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR  = "."          # 업로드한 CSV 위치 (Colab 기본). 로컬은 "data/labeled"
PRICE_DIR = "."          #   "                          로컬은 "data/unlabeled/raw"
FOLDS = {
    "A": dict(train="tsla_train_enriched_final.csv", train_ticker="TSLA",
              test="nvda_eval_final.csv",            test_ticker="NVDA",
              note="TSLA학습(보강)→NVDA평가"),
}
ENCODER = "beomi/KcELECTRA-base"

NUM_LABELS, LABELS = 4, [0, 1, 2, 3]
NAMES = ["C0예측없음", "C1실패", "C2방향적중", "C3날짜적중"]

MAX_LEN, VAL_SIZE, EPOCHS, BATCH = 512, 0.15, 6, 16
LR, WEIGHT_DECAY, WARMUP_RATIO = 1e-5, 0.01, 0.1
FOCAL_GAMMA, WEIGHT_TEMPER, SEED = 2.0, 0.5, 42
HOLDER_TOKENS = ["[주주]", "[비주주]"]

PRICE_WIN, PRICE_FEATS, PRICE_HID = 48, 4, 128   # 직전 48봉(≈7거래일)
OUT_DIR = "runs_2tower"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2. 인과적 가격 윈도우 추출기 (미래 컷오프)

종목별 시간봉을 1회 로드 → 각 댓글 `작성일` **직전** 48봉을 `searchsorted`로 슬라이스.
피처는 **윈도우-로컬 정규화**(return·(C-O)/O·(H-L)/O·거래량 z) → 전역 통계 누수도 차단.

In [ ]:
class PriceWindows:
    def __init__(self, price_dir=PRICE_DIR, win=PRICE_WIN):
        self.win = win; self.store = {}
        for tk in ("TSLA", "NVDA"):
            p = pd.read_csv(os.path.join(price_dir, f"{tk}_prices_1h.csv"),
                            encoding="utf-8-sig")
            dt = pd.to_datetime(p["Datetime_KST"])           # KST naive
            p = p.assign(_dt=dt).sort_values("_dt").reset_index(drop=True)
            self.store[tk] = (p["_dt"].values.astype("datetime64[ns]"),
                              p[["Open","High","Low","Close","Volume","return_pct"]])

    @staticmethod
    def _to_kst_naive(ts):
        return np.datetime64(pd.Timestamp(ts).tz_convert("Asia/Seoul").tz_localize(None))

    def _features(self, rows):
        o,h,l,c = (rows[k].to_numpy(float) for k in ("Open","High","Low","Close"))
        ret = np.nan_to_num(rows["return_pct"].to_numpy(float) / 100.0)
        co  = np.nan_to_num((c - o) / np.where(o == 0, np.nan, o))
        hl  = np.nan_to_num((h - l) / np.where(o == 0, np.nan, o))
        v   = np.log1p(np.clip(rows["Volume"].to_numpy(float), 0, None))
        vz  = (v - v.mean()) / (v.std() + 1e-6)
        return np.stack([ret, co, hl, vz], axis=1).astype("float32")

    def get(self, ticker, written_at):
        dts, src = self.store[ticker]
        t = self._to_kst_naive(written_at)
        end = int(np.searchsorted(dts, t, side="left"))      # strict 과거 개수
        assert end == 0 or dts[end - 1] < t, "미래 컷오프 위반"
        start = max(0, end - self.win)
        rows = src.iloc[start:end]
        seq = np.zeros((self.win, PRICE_FEATS), dtype="float32")
        mask = np.zeros((self.win,), dtype="float32")
        if len(rows) > 0:
            f = self._features(rows)
            seq[self.win - len(f):] = f                      # 뒤쪽 정렬, 앞 zero-pad
            mask[self.win - len(f):] = 1.0
        return seq, mask

## 3. 데이터셋 / 콜레이터 / 토크나이저

In [ ]:
def make_inputs(df):
    holder = df["주주_여부"].astype(str).str.lower().eq("true").map(
        {True: "[주주]", False: "[비주주]"})
    return (holder + " " + df["text"].astype(str)).tolist()

class TwoTowerDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tickers, written, tok, pw):
        self.enc = tok(texts, truncation=True, max_length=MAX_LEN)
        self.labels = list(labels); self.seqs, self.masks = [], []
        for tk, ts in zip(tickers, written):
            s, m = pw.get(tk, ts); self.seqs.append(s); self.masks.append(m)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc}
        item["price_seq"] = torch.tensor(self.seqs[i])
        item["price_mask"] = torch.tensor(self.masks[i])
        item["labels"] = int(self.labels[i])
        return item

class TwoTowerCollator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, feats):
        price_seq = torch.stack([f.pop("price_seq") for f in feats])
        price_mask = torch.stack([f.pop("price_mask") for f in feats])
        labels = torch.tensor([f.pop("labels") for f in feats], dtype=torch.long)
        batch = self.tok.pad(feats, return_tensors="pt")
        batch["price_seq"], batch["price_mask"], batch["labels"] = price_seq, price_mask, labels
        return batch

def build_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.add_special_tokens({"additional_special_tokens": HOLDER_TOKENS})
    ids = tok("테스트")["input_ids"]
    if tok.cls_token_id is not None and (len(ids) == 0 or ids[0] != tok.cls_token_id):
        from tokenizers.processors import TemplateProcessing
        tok._tokenizer.post_processor = TemplateProcessing(
            single=f"{tok.cls_token} $A {tok.sep_token}",
            pair=f"{tok.cls_token} $A {tok.sep_token} $B:1 {tok.sep_token}:1",
            special_tokens=[(tok.cls_token, tok.cls_token_id),
                            (tok.sep_token, tok.sep_token_id)])
        assert tok("테스트")["input_ids"][0] == tok.cls_token_id, "CLS 보정 실패"
        print(f"    [특수토큰 보정] {model_name}: TemplateProcessing 적용")
    return tok

## 4. 2-Tower 모델

class-weighted focal loss(γ=2.0, β=0.5)를 **forward에 내장** → 표준 `Trainer`로 macro-F1 early stopping 그대로 사용(arm A와 동일 손실).

In [ ]:
class TwoTowerClassifier(nn.Module):
    def __init__(self, encoder_name, num_labels, class_weights, focal_gamma):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        h = self.encoder.config.hidden_size
        self.price_gru = nn.GRU(PRICE_FEATS, PRICE_HID, batch_first=True, bidirectional=True)
        self.price_proj = nn.Linear(PRICE_HID * 2, PRICE_HID)
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Sequential(
            nn.Linear(h + PRICE_HID, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, num_labels))
        self.register_buffer("class_weights", class_weights)
        self.focal_gamma = focal_gamma; self.num_labels = num_labels
    def resize_token_embeddings(self, n): self.encoder.resize_token_embeddings(n)
    def _text_emb(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if getattr(out, "pooler_output", None) is not None: return out.pooler_output
        return out.last_hidden_state[:, 0]
    def _price_emb(self, price_seq, price_mask):
        seq, _ = self.price_gru(price_seq)
        m = price_mask.unsqueeze(-1)
        return self.price_proj((seq * m).sum(1) / m.sum(1).clamp(min=1.0))
    def forward(self, input_ids=None, attention_mask=None,
                price_seq=None, price_mask=None, labels=None, **kw):
        t = self._text_emb(input_ids, attention_mask)
        p = self._price_emb(price_seq, price_mask)
        logits = self.head(self.dropout(torch.cat([t, p], dim=-1)))
        loss = None
        if labels is not None:
            w = self.class_weights.to(logits.device)
            ce = F.cross_entropy(logits, labels, weight=w, reduction="none")
            if self.focal_gamma > 0:
                pt = torch.exp(-F.cross_entropy(logits, labels, reduction="none"))
                loss = ((1 - pt) ** self.focal_gamma * ce).mean()
            else:
                loss = ce.mean()
        return SequenceClassifierOutput(loss=loss, logits=logits)

## 5. 학습 + 교차평가 (Fold A: TSLA→NVDA)

In [ ]:
def metrics_fn(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "acc": accuracy_score(p.label_ids, preds)}

def report(name, y_true, y_pred):
    mf1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n  ── {name} ──  macro-F1 = {mf1:.4f}  acc = {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, labels=LABELS, target_names=NAMES,
                                digits=3, zero_division=0))
    print("  혼동행렬 (행=실제, 열=예측):\n", confusion_matrix(y_true, y_pred, labels=LABELS))
    return mf1

def load_fold(fold):
    f = FOLDS[fold]
    tr = pd.read_csv(os.path.join(DATA_DIR, f["train"]), encoding="utf-8-sig")
    te = pd.read_csv(os.path.join(DATA_DIR, f["test"]), encoding="utf-8-sig")
    for d in (tr, te):
        d.dropna(subset=["Class", "text", "작성일"], inplace=True)
        d["Class"] = d["Class"].astype(int)
    return tr, te, f

In [ ]:
def run_fold(fold, pw):
    tr_df, te_df, f = load_fold(fold)
    tok = build_tokenizer(ENCODER)
    Xtr, ytr = make_inputs(tr_df), tr_df["Class"].to_numpy()
    wtr = tr_df["작성일"].tolist(); tkr_tr = [f["train_ticker"]] * len(tr_df)
    Xte, yte = make_inputs(te_df), te_df["Class"].to_numpy()
    wte = te_df["작성일"].tolist(); tkr_te = [f["test_ticker"]] * len(te_df)
    idx = np.arange(len(Xtr))
    it, iv = train_test_split(idx, test_size=VAL_SIZE, stratify=ytr, random_state=SEED)
    g = lambda L, ix: [L[i] for i in ix]
    ds_tr = TwoTowerDS(g(Xtr, it), ytr[it], g(tkr_tr, it), g(wtr, it), tok, pw)
    ds_va = TwoTowerDS(g(Xtr, iv), ytr[iv], g(tkr_tr, iv), g(wtr, iv), tok, pw)
    ds_te = TwoTowerDS(Xte, yte, tkr_te, wte, tok, pw)
    cw = compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=ytr[it])
    cw = torch.tensor(cw, dtype=torch.float) ** WEIGHT_TEMPER
    model = TwoTowerClassifier(ENCODER, NUM_LABELS, cw, FOCAL_GAMMA)
    model.resize_token_embeddings(len(tok))
    args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, fold),
        num_train_epochs=EPOCHS, learning_rate=LR,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=32,
        eval_strategy="epoch", save_strategy="epoch", logging_steps=50,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, save_total_limit=1, seed=SEED, report_to="none",
        remove_unused_columns=False,                  # price_seq/mask 보존 필수
        fp16=torch.cuda.is_available())
    trainer = Trainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        data_collator=TwoTowerCollator(tok), compute_metrics=metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()
    pred = trainer.predict(ds_te).predictions.argmax(-1)
    return report(f"[B/2-tower] {f['note']}", yte, pred)

print(f"device={DEVICE}")
os.makedirs(OUT_DIR, exist_ok=True)
pw = PriceWindows()
print("\n===== [arm B] 2-Tower(텍스트+과거주가) — Fold A =====")
mf1 = run_fold("A", pw)
print(f"\n완료. arm B macro-F1 = {mf1:.4f}")
print("※ arm A(텍스트만)와 같은 fold/loss/metric으로 비교.")